# PROTOCOL-ARENA — SFT-only test (no Phase B RL)

Standalone notebook for evaluating an SFT-only trained model.

## When to use this

Use when the main notebook's Phase B RL hurt generalization — i.e., the post-RL model evaluated with `frame_validity ≈ 0` on held-out seeds. SFT alone tends to generalize better with small datasets / short training budgets because it imitates the diverse expert + baseline mix without the rejection-sampling noise that RL introduces.

## Pipeline

1. **Cell 1** — install + clone (~3 min)
2. **Cell 2** — generate the multi-policy SFT bootstrap (~3-5 min, CPU)
3. **Cell 3** — load Qwen2.5-1.5B 4-bit + LoRA r=16 (~2 min)
4. **Cell 4** — SFT train **2 epochs at lr=1e-4** (~25 min on T4). Stronger than the Phase A default.
5. **Cell 5** — define `sft_policy` (temperature=0.2, low sampling noise)
6. **Cell 6** — **SMOKE TEST: 1 task × 1 seed**, ~30 sec. Tells you FAST if the model produces valid actions. Aborts before the expensive full eval if not.
7. **Cell 7** — full eval across 13 tasks × 3 seeds × 3 splits (~25 min). Run only if smoke passes.
8. **Cell 8** — push the SFT-only adapter (uncomment + HF token).

Total wall time: ~60 min on free T4.

## Output

`reports/frontier_colab_sft.json` — merge into your laptop's `frontier.json`:
```bash
python scripts/merge_colab_eval.py --label trained \
    --json reports/frontier_colab_sft.json \
    --frontier reports/frontier.json
```

In [ ]:
# ---- install + clone ----
!pip -q install unsloth 'trl>=0.8' peft accelerate bitsandbytes datasets
!pip -q install openenv-core fastapi uvicorn pydantic
import os
if not os.path.exists('OpenEnv'):
    !git clone https://github.com/<YOUR-GH-USER>/OpenEnv.git
%cd OpenEnv
!pip -q install -e .

In [ ]:
# ---- generate the multi-policy SFT bootstrap ----
# Same dataset as the main notebook's Cell 2. ~3-5 min on Colab CPU.
import os
os.makedirs('data', exist_ok=True)
!python -m arena.training.sft_bootstrap \
    --out data/sft.jsonl \
    --episodes 1500 --max-per-task 120 --seed 0
!wc -l data/sft.jsonl

In [ ]:
# ---- load Qwen2.5-1.5B 4-bit + LoRA ----
from unsloth import FastLanguageModel
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
model, tok = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
tok.pad_token = tok.pad_token or tok.eos_token

In [ ]:
# ---- SFT train (2 epochs, lr=1e-4) ----
# Stronger and gentler than the main notebook's Phase A:
#   - 2 epochs (vs 1) — more passes over the same data, better generalization
#   - lr=1e-4 (vs 2e-4) — less aggressive updates, less risk of memorization
# Expected end loss: 0.10-0.30. Healthy if NOT < 0.05 (memorization).
import json, csv, os
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

os.makedirs('reports', exist_ok=True)
ds = load_dataset('json', data_files='data/sft.jsonl', split='train')
def fmt(row):
    return {'text': tok.apply_chat_template(row['messages'], tokenize=False)}
ds = ds.map(fmt)

sft_log = []
class CSVLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            sft_log.append({'step': state.global_step, 'loss': float(logs['loss'])})

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(
        output_dir='outputs/sft_only', num_train_epochs=2,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=1e-4, logging_steps=5, max_seq_length=2048,
        report_to='none', save_strategy='no',
    ),
    callbacks=[CSVLogger()],
)
trainer.train()
model.save_pretrained('outputs/sft_only'); tok.save_pretrained('outputs/sft_only')

with open('reports/sft_only_loss.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['step','loss']); w.writeheader(); w.writerows(sft_log)

print(f'\n[sft] {len(sft_log)} logged steps')
if sft_log:
    print(f'[sft] start loss: {sft_log[0]["loss"]:.3f}')
    print(f'[sft] end loss  : {sft_log[-1]["loss"]:.3f}')
    if sft_log[-1]['loss'] < 0.05:
        print('[sft] ⚠️  end loss < 0.05 — likely memorization, may not generalize')
    elif sft_log[-1]['loss'] > 0.6:
        print('[sft] ⚠️  end loss > 0.6 — model under-fit, smoke test may fail')
    else:
        print('[sft] ✅ end loss in healthy range (0.05-0.6)')

In [ ]:
# ---- define sft_policy ----
# temperature=0.2 keeps sampling tight so the model emits valid JSON
# more reliably than the main notebook's temperature=0.7.
import json
from arena.server.arena_env import ProtocolArenaEnvironment
from arena.eval.baselines import rule_based_policy
import inference

FastLanguageModel.for_inference(model)

def sft_policy(obs_dict):
    msgs = [{'role':'system', 'content': inference.SYSTEM_PROMPT},
            {'role':'user',   'content': inference.build_user_msg(obs_dict)}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inp,
        max_new_tokens=300, temperature=0.2, do_sample=True,
        pad_token_id=tok.eos_token_id,
    )
    raw = tok.decode(out[0][inp.input_ids.shape[1]:],
                     skip_special_tokens=True).strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except Exception:
        # Fallback to rule_based on invalid JSON. Counts as an invalid frame
        # in the harness, so frame_validity will reflect raw model quality.
        return rule_based_policy(obs_dict)

print('[policy] sft_policy defined with temperature=0.2')

In [ ]:
# ---- SMOKE TEST: 1 task × 1 seed (~30 sec) ----
# Tells us FAST whether the SFT-only model produces valid actions on a
# held-out task seed. If frame_validity is 0, the full 25-min eval will
# also score 0 — abort and try the honest-reporting fallback narrative.
from arena.eval.harness import run_eval

print('[smoke] running SFT-only model on research_photo_rename, seed 0...')
smoke = run_eval(sft_policy, task_ids=['research_photo_rename'], seeds=[0])

e  = smoke.get('eval_during', {})
fv = e.get('frame_validity',   {}).get('mean', 0)
tc = e.get('task_correctness', {}).get('mean', 0)
fr = e.get('final_reward',     {}).get('mean', 0)
pq = e.get('plan_quality',     {}).get('mean', 0)

print(f'\n[smoke] eval_during:')
print(f'  frame_validity   = {fv:.3f}')
print(f'  plan_quality     = {pq:.3f}')
print(f'  task_correctness = {tc:.3f}')
print(f'  final_reward     = {fr:.3f}')

if fv >= 0.5:
    print('\n✅ PASS: model produces ≥50% valid actions. Run Cell 7 for full eval.')
elif fv > 0.1:
    print('\n⚠️  MARGINAL: model produces some valid actions. Worth running full eval'
          ' but expect a noisy result. Lower temperature to 0.1 in Cell 5 and re-run'
          ' THIS smoke cell first.')
else:
    print('\n❌ FAIL: frame_validity ≈ 0 — every action rejected. SFT-only also'
          ' broken on held-out seeds. Skip Cell 7. Use honest-reporting narrative'
          ' (see TRACK_A_TRAINER.md SFT-only fallback section).')

In [ ]:
# ---- FULL EVAL — only run if Cell 6 smoke PASSED ----
# Trained (SFT-only) + 3 baselines, across 13 tasks × 3 seeds × 3 splits.
# Frontier APIs (gpt-4o-mini, claude-haiku-4-5) get added later by the
# laptop's run_frontier.py + merge_colab_eval.py pipeline.
from arena.eval.harness import run_eval
from arena.eval.baselines import rule_based_policy, keyword_policy, random_policy
import json

print('[eval] full sweep — ~25 min on T4')
trained_report = run_eval(sft_policy, seeds=[0, 1, 2])
rule_report    = run_eval(rule_based_policy, seeds=[0, 1, 2])
keyword_report = run_eval(keyword_policy, seeds=[0, 1, 2])
random_report  = run_eval(random_policy, seeds=[0, 1, 2])

out = {'providers': {
    'trained':    trained_report,    # ← this is the SFT-only model
    'rule_based': rule_report,
    'keyword':    keyword_report,
    'random':     random_report,
}}
with open('reports/frontier_colab_sft.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)

# Quick verdict print
e = trained_report['eval_during']
print(f'\n[eval] trained eval_during:')
print(f'  task_correctness = {e["task_correctness"]["mean"]:.3f}')
print(f'  frame_validity   = {e["frame_validity"]["mean"]:.3f}')
print(f'  plan_quality     = {e["plan_quality"]["mean"]:.3f}')
print(f'  final_reward     = {e["final_reward"]["mean"]:.3f}')
print(f'\n[eval] wrote reports/frontier_colab_sft.json')
print(f'[eval] On your laptop:')
print(f'  python scripts/merge_colab_eval.py \\')
print(f'      --label trained \\')
print(f'      --json reports/frontier_colab_sft.json \\')
print(f'      --frontier reports/frontier.json')

In [ ]:
# ---- push SFT-only adapter to HF Hub (uncomment when ready) ----
# Only push if Cell 6 smoke passed AND Cell 7 final_reward >= 0.45.
# Otherwise the published adapter will look broken to anyone evaluating it.
#
# from huggingface_hub import notebook_login; notebook_login()
# REPO = 'YOUR-ORG/protocol-arena-qwen-1.5b-lora-r16-sft-only'
# model.push_to_hub(REPO)
# tok.push_to_hub(REPO)
# print(f'[push] adapter at https://huggingface.co/{REPO}')
# print(f'[push] hand this URL to Track C for the README')